# EDA RA 2025
Análisis exploratorio reproducible. Los cocientes calculados no son tasas oficiales y ninguna asociación se interpreta causalmente.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from src.analysis.ra_2025 import (write_analytical_table, write_variable_inventory, aggregate_departments, add_structural_shares)
OUT = Path('outputs/charts/eda')
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
analitico = write_analytical_table()
inventario = write_variable_inventory()
departamentos = add_structural_shares(analitico, aggregate_departments(analitico))
assert analitico.departamento_id.nunique() == 506
analitico[['cobertura_matricula','cobertura_trayectoria','cobertura_caracteristicas']].value_counts()

In [ ]:
metricas = ['proporcion_repetidores_sobre_matricula','proporcion_sobreedad_sobre_matricula','proporcion_no_promovidos_sobre_ultimo','proporcion_salidos_sin_pase_sobre_inicial']
departamentos[metricas].describe().T

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, variable in zip(axes.flat, metricas):
    ax.hist(departamentos[variable].dropna(), bins=30)
    ax.set_title(variable.replace('proporcion_', '').replace('_', ' '))
fig.tight_layout(); fig.savefig(OUT/'distribuciones_indicadores.png', dpi=150); plt.show()

In [ ]:
pares = departamentos[['proporcion_repetidores_sobre_matricula','proporcion_sobreedad_sobre_matricula']].dropna()
ax = pares.plot.scatter(x=pares.columns[0], y=pares.columns[1], alpha=.45)
ax.figure.tight_layout(); ax.figure.savefig(OUT/'repeticion_y_sobreedad.png', dpi=150); plt.show()
pares.corr(), len(pares)

In [ ]:
heterogeneidad = departamentos.groupby('provincia_nombre').agg(n=('departamento_id','size'), iqr_sobreedad=('proporcion_sobreedad_sobre_matricula', lambda x: x.quantile(.75)-x.quantile(.25)))
heterogeneidad.query('n >= 10').sort_values('iqr_sobreedad', ascending=False)

## Conclusiones provisionales
Los hallazgos auditados, sus coberturas y límites de interpretación se documentan en `docs/analisis/eda_ra_2025.md`. Este cuaderno no reemplaza esa ficha metodológica.